# Exploratory data analysis

The goal of this notebook is to profile the data and see if any data cleaning or pre processing steps are required.

## Setting up data from azure

Core idea of this layer: adress all the potential issues flagged by the data profiling library.

In [ ]:
import os
import sys
import time
sys.path.append("..")

import polars as pl
from data_profiling import ProfileReport
from ingest import get_engine

In [ ]:
engine = get_engine()

print("Waking up Azure SQL...")
for attempt in range(1, 11):
    try:
        with engine.connect() as conn:
            conn.exec_driver_sql("SELECT 1")
        print("Database is ready.")
        break
    except Exception:
        print(f"Attempt {attempt}/10 — retrying in 10s...")
        time.sleep(10)
else:
    raise RuntimeError("Database did not respond after 10 attempts.")

In [ ]:
customers            = pl.read_database("SELECT * FROM olist_customers_dataset", engine)
geolocation          = pl.read_database("SELECT * FROM olist_geolocation_dataset", engine)
order_items          = pl.read_database("SELECT * FROM olist_order_items_dataset", engine)
order_payments       = pl.read_database("SELECT * FROM olist_order_payments_dataset", engine)
order_reviews        = pl.read_database("SELECT * FROM olist_order_reviews_dataset", engine)
orders               = pl.read_database("SELECT * FROM olist_orders_dataset", engine)
products             = pl.read_database("SELECT * FROM olist_products_dataset", engine)
sellers              = pl.read_database("SELECT * FROM olist_sellers_dataset", engine)
category_translation = pl.read_database("SELECT * FROM product_category_name_translation", engine)

In [ ]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation,
}

In [ ]:
def generate_table_info(datasets, info_dir="info"):
    os.makedirs(info_dir, exist_ok=True)

    if os.listdir(info_dir):
        print(f"'{info_dir}' is not empty, skipping generation.")
        return

    tables_summary = []

    for name, df in datasets.items():
        lines = [
            f"Dataset: {name}",
            f"Shape: {df.shape[0]} rows x {df.shape[1]} columns",
            "Columns:",
            *(f"  - {col} ({dtype})" for col, dtype in zip(df.columns, df.dtypes)),
        ]
        content = "\n".join(lines)

        with open(f"{info_dir}/{name}.txt", "w") as f:
            f.write(content + "\n")

        tables_summary.append(content)
        print(f"Generated: {info_dir}/{name}.txt")

    with open(f"{info_dir}/tables.txt", "w") as f:
        f.write("\n\n".join(tables_summary) + "\n")

    print(f"Generated: {info_dir}/tables.txt")


generate_table_info(datasets)

In [ ]:
def generate_html_reports(datasets, reports_dir="html_reports"):
    os.makedirs(reports_dir, exist_ok=True)

    if os.listdir(reports_dir):
        print(f"'{reports_dir}' is not empty, skipping generation.")
        return

    for name, df in datasets.items():
        report = ProfileReport(df.to_pandas(), title=f"{name} profiling report", minimal=True, progress_bar=False) # minimal generation does not account for duplicates
        report.to_file(f"{reports_dir}/{name}.html")
        print(f"Generated: {reports_dir}/{name}.html")


generate_html_reports(datasets)


## Bronze to Silver

### Customers

Preprocessing: drop customer_unique_id column.

In [ ]:
customers.shape

In [ ]:
customers.head()

In [ ]:
customers['customer_unique_id'].is_unique().all()

In [ ]:
customers.describe()

Since the customers_unique_id is actually not unique, we will drop this column.

In [ ]:
customers = customers.drop("customer_unique_id")

In [ ]:
customers['customer_id'].is_unique().all()

### Geolocation

Pre-processing: grouping by the zip code to reduce duplicates.

Geolocation dataset has no unique column, although it is connected via a zip_code_prefix to the customers dataset. We want to make that column unique.

In [ ]:
geolocation.shape

In [ ]:
geolocation.head()

In [ ]:
geolocation.null_count()

In [ ]:
geolocation = geolocation.group_by("geolocation_zip_code_prefix").agg(
    pl.col("geolocation_lat").mean(),
    pl.col("geolocation_lng").mean(),
    pl.col("geolocation_city").mode().first(),
    pl.col("geolocation_state").mode().first(),
)

geolocation

The aggregation reduced the dataset size from 1M rows to 19K! There were a lot of duplicates in the data, potentially the artifact of order processing. 

In [ ]:
geolocation["geolocation_zip_code_prefix"].is_unique().all()

In [ ]:
geolocation = geolocation.sort("geolocation_zip_code_prefix", descending=False)

### Order items

Pre-processing: conversion to timestamp data

Order item id represents the position of an item within a basket. A single customer can order multiple items e.g 3, so then for that specific order_id we would have 3 order item id's.

In [ ]:
order_items.head()

In [ ]:
order_items.null_count()

In [ ]:
order_items = order_items.with_columns(
    pl.col("shipping_limit_date").str.to_datetime("%Y-%m-%d %H:%M:%S")
)

This seems like an intermediate table, that would be useful in the data agreggation within the gold layer.

In [ ]:
order_items.head()

### Order payments

A good sanity check can be to verify that the order sum from order payments matches the price in the order_items & orders tables! 

In [ ]:
order_payments.head()

In [ ]:
order_payments.null_count()

In [ ]:
order_payments.filter(pl.col("payment_sequential") > 5).sort('payment_sequential', descending=True)

### Order reviews

Pre-processing: drop the review title and the review comment messages, and convert the timestamp data.

In [ ]:
order_reviews.head()

In [ ]:
order_reviews['review_id'].is_unique().all()

Interestingly not all review id's are completely unique

In [ ]:
order_reviews.null_count() / order_reviews.height

More than 50% of the data is missing for the review text columns. Since we are not concerned with NLP we will drop those columns. We will also convert timestamp data into timestamp format.

In [ ]:
order_reviews = (
    order_reviews
    .drop(["review_comment_title", "review_comment_message"])
    .with_columns(
        pl.col("review_creation_date").str.to_datetime("%Y-%m-%d %H:%M:%S"),
        pl.col("review_answer_timestamp").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    )
)
order_reviews

In [ ]:
order_reviews.filter(order_reviews['review_id'].is_unique() == False).sort('review_id')

Same review can occur for multiple orders! When joining this dataset it is better to use review_id + order_id as a composite key.

### Orders

Pre processing: conversion of the timestamp data to proper type.

In [ ]:
orders.head()

Order id and customer id are both unique columns

In [ ]:
orders["order_id"].is_unique().all() and orders["customer_id"].is_unique().all()

There are however some missing values for carrier date and customer date. The delivery pipeline works as follows: 

purchased → payment approved → handed to carrier → delivered to customer → (compared against) estimated delivery date

Missing values are thus informative of underperformance that could be the culprit for the seller under-performance, lowering the sales volume.

If the carrier date is null, then that means that the seller never handed off the delivery to the carrier, while the customer date null means that the order was never delivered to the customer.

In [ ]:
orders.filter(pl.col("order_delivered_customer_date").is_null()).select(['order_delivered_carrier_date', 'order_delivered_customer_date'])

In [ ]:
orders.filter(
    pl.col("order_delivered_carrier_date").is_null()
    & pl.col("order_delivered_customer_date").is_not_null()
)

Majority of the orders are handled through a carrier, although a single case shows a situation where the order was delivered directly to the customer, bypassing the carrier.

There could be several explanations:

Either the seller delivered the item directly, or this is a logging error. 

In any case a single row will not move meaningfully any aggregate metric, although it is useful to be aware about potential issues.

In [ ]:
orders.group_by("order_status").agg(
    pl.col("order_delivered_carrier_date").null_count().alias("carrier_nulls"),
    pl.col("order_delivered_customer_date").null_count().alias("customer_nulls"),
    pl.len().alias("total"),
)

We have some minor inconsistencies, which are probably just logging errors, they will not meaningfully affect the results. In general cases the null values are representative - e.g if the order is shipped, it should hot have a customer delivery date. 

In [ ]:
orders = orders.with_columns(
    pl.col("order_purchase_timestamp").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("order_approved_at").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("order_delivered_carrier_date").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("order_delivered_customer_date").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col("order_estimated_delivery_date").str.to_datetime("%Y-%m-%d %H:%M:%S"),
)

In [ ]:
orders.head()

### Products

For the products category in the gold layer it might be better to swap the category names to the english version.

In [ ]:
products.head()

In [ ]:
products.filter(pl.col('product_category_name').is_null())

In [ ]:
products.null_count() / products.height

Even though we could drop the nulls for the product category name, the unique product_id's might come in handy when joining the tables. We will leave them in the pre processing pipeline.

### Sellers

No preprocesssing required here.

In [ ]:
sellers.head()

### Category translation

No preprocessing required here

In [ ]:
category_translation.head()

## Silver to Gold - Reshaping the data for business analysis (Power BI)

In [ ]:
customers_cleaned            = customers
geolocation_cleaned          = geolocation
order_items_cleaned          = order_items
order_payments_cleaned       = order_payments
order_reviews_cleaned        = order_reviews
orders_cleaned               = orders
products_cleaned             = products
sellers_cleaned              = sellers
category_translation_cleaned = category_translation

Core idea of this layer: remove data redundancy and reshape the data into a star schema to support the business analysis.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Star schema: one atomic fact (fact_sales) + surrounding dimensions.
# The fact holds ONLY foreign keys + additive measures — every average,
# distribution and rate is left for Power BI to compute from these tables.
# ══════════════════════════════════════════════════════════════════════

In [ ]:
# ── fact_sales ── grain: one order item ───────────────────────────────
# order_items is already at item grain; we only bring the order'''s purchase
# date (-> date_key) and customer onto each line so the date and customer
# dimensions can connect. left join keeps every item row, so grain holds.
fact_sales = (
    order_items_cleaned
    .join(
        orders_cleaned.select("order_id", "customer_id", "order_purchase_timestamp"),
        on="order_id",
        how="left",
    )
    .with_columns(
        # integer YYYYMMDD surrogate key into dim_date
        pl.col("order_purchase_timestamp").dt.strftime("%Y%m%d").cast(pl.Int32)
        .alias("order_purchase_date_key"),
        # total amount paid for the line item (item price + shipping fee)
        (pl.col("price") + pl.col("freight_value")).alias("total_cost"),
    )
    .select(
        "order_id",                  # FK -> dim_order  /  FK -> dim_payments
        "seller_id",                 # FK -> dim_seller
        "product_id",                # FK -> dim_product
        "customer_id",               # FK -> dim_customer
        "order_purchase_date_key",   # FK -> dim_date
        "price",                     # measure
        "freight_value",             # measure
        "total_cost",                # measure: price + freight_value
    )
)

In [ ]:
fact_sales.head()

In [ ]:
# ── dim_seller ── one row per seller ──────────────────────────────────
# seller_zip_code_prefix joins to dim_geolocation for lat/lng.
dim_seller = sellers_cleaned  # seller_id, zip_code_prefix, city, state

In [ ]:
dim_seller.head()

In [ ]:
dim_seller['seller_id'].is_unique().all()

In [ ]:
# ── dim_product ── one row per product, category translated to English ─
dim_product = (
    products_cleaned
    .join(category_translation_cleaned, on="product_category_name", how="left")
    .select(
        "product_id",
        "product_category_name_english",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    )
)

In [ ]:
dim_product.head()

In [ ]:
dim_product['product_id'].is_unique().all()

In [ ]:
# ── dim_customer ── geography per customer_id (1:1 with an order) ──────
# customer_zip_code_prefix joins to dim_geolocation for lat/lng.
dim_customer = customers_cleaned.select(
    "customer_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
)


In [ ]:
dim_customer.head()

In [ ]:
dim_customer['customer_id'].is_unique().all()

In [ ]:
# ── dim_order ── order-level "why" attributes + one review per order ──
# Reviews are NOT unique per order, so collapse to the latest review score
# first — otherwise the join would fan out (duplicate) the order rows.
order_review = (
    order_reviews_cleaned
    .sort("review_creation_date")
    .group_by("order_id")
    .agg(pl.col("review_score").last())
)
dim_order = (
    orders_cleaned
    .join(order_review, on="order_id", how="left")
    .select(
        "order_id",
        "order_status",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "review_score",
    )
)

In [ ]:
dim_order.head()

In [ ]:
dim_order['order_id'].is_unique().all()

In [ ]:
# ── dim_payments ── one row per order (payment rows rolled up) ────────
# An order can be split across payment methods (payment_sequential 1..n).
# Keep the primary method (sequential = 1) + the total value paid.
dim_payments = (
    order_payments_cleaned
    .sort("payment_sequential")
    .group_by("order_id")
    .agg(
        pl.col("payment_type").first().alias("primary_payment_type"),                 # primary method
        pl.col("payment_installments").first().alias("primary_payment_installments"), # of primary method
        pl.col("payment_value").sum().alias("payment_value"),                 # total across methods
        pl.len().alias("payment_methods_count"),                              # how many methods used
    )
)

In [ ]:
dim_payments.head()

In [ ]:
dim_payments['order_id'].is_unique().all()

In [ ]:
# ── dim_date ── contiguous calendar spanning the order purchase dates ─
# A full date spine (not just the dates that appear) so Power BI can mark
# it as a Date Table and build a clean year > quarter > month hierarchy.
_bounds = orders_cleaned.select(
    pl.col("order_purchase_timestamp").dt.date().min().alias("start"),
    pl.col("order_purchase_timestamp").dt.date().max().alias("end"),
)
dim_date = (
    pl.select(
        pl.date_range(_bounds["start"][0], _bounds["end"][0], interval="1d").alias("full_date")
    )
    .with_columns(
        pl.col("full_date").dt.strftime("%Y%m%d").cast(pl.Int32).alias("date_key"),
        pl.col("full_date").dt.year().alias("year"),
        pl.col("full_date").dt.quarter().alias("quarter"),
        pl.col("full_date").dt.month().alias("month"),
        pl.col("full_date").dt.strftime("%B").alias("month_name"),
    )
    .with_columns(
        # contiguous month counter from the start of the date range, so
        # Power BI can compare/sort months across year boundaries 
        ((pl.col("year") - pl.col("year").min()) * 12 + pl.col("month") - 9).alias("month_index")
    )
    .select("date_key", "full_date", "year", "quarter", "month", "month_name", "month_index")
)


In [ ]:
dim_date.head()

In [ ]:
dim_date['month_index'].max()

In [ ]:
dim_date.sort('full_date', descending=False).head()

In [ ]:
dim_date['date_key'].is_unique().all()

### Geolocation

In [ ]:
# ── dim_geolocation ── one row per zip prefix, shared by seller + customer ─
# geolocation_cleaned is already deduplicated to one row per
# geolocation_zip_code_prefix (see Bronze -> Silver step). dim_seller and
# dim_customer each carry their own zip_code_prefix column, so both relate
# to this single shared dimension in Power BI without duplicating lat/lng.
dim_geolocation = geolocation_cleaned.rename({"geolocation_zip_code_prefix": "zip_code_prefix"})


In [ ]:
dim_geolocation.head()

In [ ]:
dim_geolocation['zip_code_prefix'].is_unique().all()

### Seller-month revenue

In [ ]:
# ── fact_seller_month ── grain: one row per (seller, month) ────────────
fact_seller_month = (
    fact_sales
    .join(dim_date.select("date_key", "month_index"), left_on="order_purchase_date_key", right_on="date_key", how="left")
    .group_by("seller_id", "month_index")
    .agg(pl.col("total_cost").sum().alias("revenue"))
    .sort("seller_id", "month_index")
)

In [ ]:
fact_seller_month.head(10)

In [ ]:
fact_seller_month.select("seller_id", "month_index").is_unique().all()

In [ ]:
# ── dim_seller enrichment ── months_active ─────────────────────────────
months_active = (
    fact_seller_month
    .group_by("seller_id")
    .agg(pl.col("month_index").n_unique().alias("months_active"))
)

dim_seller = dim_seller.join(months_active, on="seller_id", how="left")

In [ ]:
dim_seller.head()

In [ ]:
dim_seller['seller_id'].is_unique().all()

In [ ]:
fact_seller_month['month_index'].value_counts().sort('month_index')

### Why the band is trimmed to month_index 4–23

`month_index` runs 0–25 across the full dataset (Sep 2016 – Oct 2018), but the
edges aren't real trading activity — they're where Olist's order history
happens to start/stop mid-stream:

| month_index | active sellers | total revenue |
|---|---|---|
| 0 | 3 | R$355 |
| 1 | 143 | R$56,809 |
| 3 | 1 | R$20 |
| 4 | 227 | R$137,188 |
| ... | (200–1,278 sellers, R$130K–1.16M/month) | ... |
| 23 | 1,278 | R$1,003,308 |
| 24 | 1 | R$166 |

(month 2 has no rows at all.) Months 0, 1, 3, and 24 have only 1–3 sellers and
near-zero revenue — a handful of orders that happened to land just outside the
platform's real operating window, not a genuine ramp-up or ramp-down in seller
behavior. Fitting the OLS slope across those months would let a seller's score
be dominated by whether they have a data point in a month where almost nobody
else does, rather than by their actual trend. Trimming to 4–23 keeps the fit
inside the ~20-month band where monthly activity is consistently in the
hundreds of sellers, so the slope reflects real revenue trajectory.

In [ ]:
# ── dim_seller enrichment ── revenue_risk_score ───────────────────────
# Normalized hand-written OLS slope of monthly revenue over month_index.
# Natural slope sign kept: negative = revenue declining (at risk), positive
# = growing; the more negative, the faster the decline. Fit on the clean
# operating band (month_index 4..23), observed months only, for sellers with
# >= 12 months of data in that band. Others -> null.
MIN_MONTHS = 12
BAND_LO, BAND_HI = 4, 23

risk = (
    fact_seller_month
    .filter(pl.col("month_index").is_between(BAND_LO, BAND_HI))     # trim edges
    .group_by("seller_id")
    .agg(
        pl.len().alias("n"),
        pl.col("month_index").sum().alias("sx"),
        pl.col("revenue").sum().alias("sy"),
        (pl.col("month_index") * pl.col("revenue")).sum().alias("sxy"),
        (pl.col("month_index") ** 2).sum().alias("sxx"),
        pl.col("revenue").mean().alias("mean_rev"),
    )
    .with_columns(
        # OLS slope b = (n·Σxy − Σx·Σy) / (n·Σxx − (Σx)²)
        (
            (pl.col("n") * pl.col("sxy") - pl.col("sx") * pl.col("sy"))
            / (pl.col("n") * pl.col("sxx") - pl.col("sx") ** 2)
        ).alias("slope")
    )
    .with_columns(
        # normalize by mean revenue, keep natural sign: negative = declining
        pl.when((pl.col("n") >= MIN_MONTHS) & (pl.col("mean_rev") > 0))
          .then(pl.col("slope") / pl.col("mean_rev"))
          .otherwise(None)
          .alias("revenue_risk_score")
    )
    .select("seller_id", "revenue_risk_score")
)

dim_seller = dim_seller.join(risk, on="seller_id", how="left")

In [ ]:
dim_seller.head()

In [ ]:
dim_seller['seller_id'].is_unique().all()

In [ ]:
# eligible cohort + most at-risk sellers (most negative score first)
print("scored sellers:", dim_seller.filter(pl.col("revenue_risk_score").is_not_null()).height)
dim_seller.filter(pl.col("revenue_risk_score").is_not_null()).sort("revenue_risk_score").head(10)

### File export

In [ ]:
gold_tables = {
    "fact_sales": fact_sales,
    "fact_seller_month": fact_seller_month,
    "dim_seller": dim_seller,
    "dim_product": dim_product,
    "dim_customer": dim_customer,
    "dim_order": dim_order,
    "dim_payments": dim_payments,
    "dim_date": dim_date,
    "dim_geolocation": dim_geolocation,
}

export_dir = "power_bi_exports"
os.makedirs(export_dir, exist_ok=True)

for name, df in gold_tables.items():
    df.write_csv(f"{export_dir}/{name}.csv")
    print(f"Exported: {export_dir}/{name}.csv")